# NeuTherm-PINN: Project Walkthrough

This notebook walks through the complete pipeline:
1. **Reference Solver** — Coupled neutronics-thermal solution via Picard iteration
2. **Surrogate Model** — Data-driven FNN trained on 5000 solver samples
3. **PINN** — Physics-informed network learning from governing equations
4. **Comparison** — Side-by-side error analysis

> Requires a trained surrogate (`results/surrogate_model.pt`) and PINN (`results/pinn_model.pt`).  
> If you haven't trained them yet, run the CLI commands in the README first.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from neutherm.physics.parameters import ProblemConfig
from neutherm.solvers.coupled_solver import solve_coupled
from neutherm.evaluation.metrics import relative_l2, pointwise_relative_error

# Load configuration
config = ProblemConfig.from_yaml("configs/default.yaml")
config.validate()
print("Configuration loaded successfully.")

## 1. Reference Solver

The coupled solver alternates between:
- **Neutron diffusion** (2-group, finite differences + power iteration)
- **Heat conduction** (radial, temperature-dependent UO₂ conductivity)

until both $k_{\text{eff}}$ and the temperature field converge (Picard iteration).

In [ ]:
# Run the reference solver
solution = solve_coupled(config, power_level=200.0, verbose=True)

In [ ]:
# Plot the reference solution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

n_fuel = config.geometry.n_radial
r_fuel = solution.r_fuel
r_full = solution.r_neutronics
R_fuel = r_fuel[-1]

# Neutron fluxes (full pin cell)
axes[0].plot(r_full, solution.phi1, label="Fast (group 1)")
axes[0].plot(r_full, solution.phi2, label="Thermal (group 2)")
axes[0].axvline(R_fuel, color="gray", ls="--", lw=0.8, label="Fuel/mod boundary")
axes[0].set_xlabel("r [cm]")
axes[0].set_ylabel("Neutron flux [a.u.]")
axes[0].set_title(f"Neutron flux ($k_{{eff}}$ = {solution.k_eff:.5f})")
axes[0].legend()

# Temperature (fuel only)
axes[1].plot(r_fuel, solution.temperature, color="tab:red")
axes[1].set_xlabel("r [cm]")
axes[1].set_ylabel("Temperature [K]")
axes[1].set_title(f"Fuel temperature (ΔT = {solution.temperature[0]-solution.temperature[-1]:.0f} K)")

# Heat generation (fuel only)
axes[2].plot(r_fuel, solution.q_volumetric / 1e6, color="tab:orange")
axes[2].set_xlabel("r [cm]")
axes[2].set_ylabel("q''' [MW/m³]")
axes[2].set_title("Volumetric heat generation")

plt.tight_layout()
plt.show()

print(f"\nk_eff = {solution.k_eff:.6f}")
print(f"T_centerline = {solution.temperature[0]:.1f} K")
print(f"T_surface = {solution.temperature[-1]:.1f} K")

## 2. Surrogate Model

The surrogate is a feed-forward neural network (FNN) with residual blocks that learns:

$$\mathcal{F}_\theta : (T_{\text{coolant}},\; R_f,\; \text{enrichment}) \mapsto (\phi_1(r),\; \phi_2(r),\; T(r),\; k_{\text{eff}})$$

Trained on 5000 solver solutions generated via Latin Hypercube Sampling.

In [ ]:
from neutherm.models.surrogate import SurrogateModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained surrogate — architecture comes from the checkpoint when
# available (older checkpoints fall back to the YAML config)
checkpoint = torch.load("results/surrogate_model.pt", map_location=device, weights_only=False)
mc = checkpoint["model_config"]

surr_model = SurrogateModel(
    n_inputs=mc["n_inputs"],
    n_radial_fuel=mc["n_radial_fuel"],
    n_radial_total=mc["n_radial_total"],
    hidden_layers=mc.get("hidden_layers", config.surrogate.hidden_layers),
    activation=mc.get("activation", config.surrogate.activation),
).to(device)

# Initialize normalizer then load weights
surr_model.set_normalizer(
    torch.zeros(mc["n_inputs"], device=device),
    torch.ones(mc["n_inputs"], device=device),
)
surr_model.load_state_dict(checkpoint["model_state_dict"])
surr_model.eval()

ns = checkpoint.get("norm_stats", None)
print(f"Surrogate loaded: {surr_model.count_parameters():,} parameters")


In [ ]:
# Predict with the surrogate for the default config
params = torch.tensor([[
    config.thermal.T_coolant,
    config.geometry.r_fuel,
    1.0,  # enrichment factor
]], dtype=torch.float32).to(device)

with torch.no_grad():
    pred = surr_model(params)

# Denormalize (training stores per-point mean/std of each output field)
if ns is not None:
    surr_phi1 = pred["phi1"].cpu().numpy()[0] * ns["phi1_std"].cpu().numpy() + ns["phi1_mean"].cpu().numpy()
    surr_phi2 = pred["phi2"].cpu().numpy()[0] * ns["phi2_std"].cpu().numpy() + ns["phi2_mean"].cpu().numpy()
    surr_temp = pred["temperature"].cpu().numpy()[0] * ns["temp_std"].cpu().numpy() + ns["temp_mean"].cpu().numpy()
    surr_keff = float(pred["k_eff"].cpu().numpy()[0,0] * ns["keff_std"].cpu().numpy() + ns["keff_mean"].cpu().numpy())
else:
    surr_phi1 = pred["phi1"].cpu().numpy()[0]
    surr_phi2 = pred["phi2"].cpu().numpy()[0]
    surr_temp = pred["temperature"].cpu().numpy()[0]
    surr_keff = float(pred["k_eff"].cpu().numpy()[0,0])

print(f"Surrogate k_eff = {surr_keff:.6f}")
print(f"Solver k_eff    = {solution.k_eff:.6f}")
print(f"Relative error  = {abs(surr_keff - solution.k_eff)/solution.k_eff*100:.4f}%")


## 3. PINN

The Physics-Informed Neural Network approximates the solution as a continuous function:

$$\text{NN}(r) \to (\phi_1(r),\; \phi_2(r),\; T(r))$$

The PDE residuals are computed via `torch.autograd` and penalized in the loss function. $k_{\text{eff}}$ is learned as an additional trainable parameter.

In [ ]:
from neutherm.models.pinn import PINNModel

# Load trained PINN — architecture and physical scales come from the
# checkpoint (with fallbacks for legacy checkpoints)
pinn_ckpt = torch.load("results/pinn_model.pt", map_location=device, weights_only=False)

pinn_model = PINNModel(
    hidden_layers=pinn_ckpt.get("hidden_layers", config.pinn.hidden_layers),
    activation=pinn_ckpt.get("activation", config.pinn.activation),
    r_fuel=pinn_ckpt.get("r_fuel", config.geometry.r_fuel * 100),
    r_cell=pinn_ckpt.get("r_cell", config.geometry.r_cell * 100),
    T_out_scale=pinn_ckpt.get("T_out_scale", 1.0),
).to(device)

pinn_model.load_state_dict(pinn_ckpt["model_state_dict"])
pinn_model.eval()

pinn_keff = pinn_ckpt["k_eff"]

# phi_scale converts the network's O(1) "shape" flux into the physical,
# power-normalized flux; T_base anchors the temperature head (it equals
# the fuel surface temperature computed from the linear power).
pinn_phi_scale = pinn_ckpt.get("phi_scale", 1.0)
T_base = pinn_ckpt.get("T_base", float(config.thermal.T_coolant + 300.0))

print(f"PINN loaded: {pinn_model.count_parameters():,} parameters")
print(f"PINN k_eff = {pinn_keff:.6f}")
print(f"phi_scale  = {pinn_phi_scale:.3e}   T_base = {T_base:.1f} K")


## 4. Comparison: Solver vs Surrogate vs PINN

In [ ]:
# Predict PINN: fluxes on the full pin-cell mesh, temperature on the fuel mesh
with torch.no_grad():
    r_all_t = torch.tensor(r_full, dtype=torch.float32).unsqueeze(1).to(device)
    out_all = pinn_model(r_all_t)
    pinn_phi1 = pinn_phi_scale * out_all["phi1"].cpu().numpy().squeeze()
    pinn_phi2 = pinn_phi_scale * out_all["phi2"].cpu().numpy().squeeze()

    r_fuel_t = torch.tensor(r_fuel, dtype=torch.float32).unsqueeze(1).to(device)
    pinn_temp = T_base + pinn_model(r_fuel_t)["T"].cpu().numpy().squeeze()

# Comparison plot
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Row 1: Profiles — fluxes over the FULL pin cell (fuel + moderator)
axes[0,0].plot(r_full, solution.phi1, "k-", lw=2, label="Solver")
axes[0,0].plot(r_full, surr_phi1, "b--", lw=1.5, label="Surrogate")
axes[0,0].plot(r_full, pinn_phi1, "r:", lw=1.5, label="PINN")
axes[0,0].axvline(R_fuel, color="gray", ls="--", lw=0.8)
axes[0,0].set_xlabel("r [cm]"); axes[0,0].set_ylabel("φ₁ [n/cm²·s]")
axes[0,0].set_title("Fast-group flux"); axes[0,0].legend()

axes[0,1].plot(r_full, solution.phi2, "k-", lw=2, label="Solver")
axes[0,1].plot(r_full, surr_phi2, "b--", lw=1.5, label="Surrogate")
axes[0,1].plot(r_full, pinn_phi2, "r:", lw=1.5, label="PINN")
axes[0,1].axvline(R_fuel, color="gray", ls="--", lw=0.8)
axes[0,1].set_xlabel("r [cm]"); axes[0,1].set_ylabel("φ₂ [n/cm²·s]")
axes[0,1].set_title("Thermal-group flux"); axes[0,1].legend()

axes[0,2].plot(r_fuel, solution.temperature, "k-", lw=2, label="Solver")
axes[0,2].plot(r_fuel, surr_temp, "b--", lw=1.5, label="Surrogate")
axes[0,2].plot(r_fuel, pinn_temp, "r:", lw=1.5, label="PINN")
axes[0,2].set_xlabel("r [cm]"); axes[0,2].set_ylabel("T [K]")
axes[0,2].set_title("Temperature"); axes[0,2].legend()

# Row 2: Pointwise errors
for i, (name, r_ax, surr_f, pinn_f, ref_f) in enumerate([
    ("Fast flux", r_full, surr_phi1, pinn_phi1, solution.phi1),
    ("Thermal flux", r_full, surr_phi2, pinn_phi2, solution.phi2),
    ("Temperature", r_fuel, surr_temp, pinn_temp, solution.temperature),
]):
    surr_err = pointwise_relative_error(surr_f, ref_f)
    pinn_err = pointwise_relative_error(pinn_f, ref_f)
    axes[1,i].semilogy(r_ax, surr_err, "b-", label="Surrogate")
    axes[1,i].semilogy(r_ax, pinn_err, "r-", label="PINN")
    if name != "Temperature":
        axes[1,i].axvline(R_fuel, color="gray", ls="--", lw=0.8)
    axes[1,i].set_xlabel("r [cm]"); axes[1,i].set_ylabel("Relative error")
    axes[1,i].set_title(f"{name} — pointwise error"); axes[1,i].legend()

plt.tight_layout()
plt.savefig("results/comparison.png", dpi=150)
plt.show()
print("Comparison plot saved to results/comparison.png")


In [ ]:
# Summary table (fluxes compared on the full pin cell, T in the fuel)
print("=" * 60)
print(f"{'Metric':<25s} {'Surrogate':>15s} {'PINN':>15s}")
print("-" * 60)
print(f"{'k_eff':<25s} {surr_keff:>15.6f} {pinn_keff:>15.6f}")
print(f"{'k_eff (solver)':<25s} {solution.k_eff:>15.6f} {solution.k_eff:>15.6f}")

surr_keff_err = abs(surr_keff - solution.k_eff) / solution.k_eff
pinn_keff_err = abs(pinn_keff - solution.k_eff) / solution.k_eff
print(f"{'k_eff rel. error':<25s} {surr_keff_err:>14.4%} {pinn_keff_err:>14.4%}")

print(f"{'φ₁ rel. L2':<25s} {relative_l2(surr_phi1, solution.phi1):>14.4%} {relative_l2(pinn_phi1, solution.phi1):>14.4%}")
print(f"{'φ₂ rel. L2':<25s} {relative_l2(surr_phi2, solution.phi2):>14.4%} {relative_l2(pinn_phi2, solution.phi2):>14.4%}")
print(f"{'T rel. L2':<25s} {relative_l2(surr_temp, solution.temperature):>14.4%} {relative_l2(pinn_temp, solution.temperature):>14.4%}")
print("-" * 60)
print(f"{'Training data':<25s} {'5000 samples':>15s} {'1 ref. solve':>15s}")
print(f"{'Parameters':<25s} {surr_model.count_parameters():>15,d} {pinn_model.count_parameters():>15,d}")
print("=" * 60)


## Key Takeaways

1. **Surrogate model** reproduces the reference solver almost exactly across the sampled parameter range, but needs thousands of labeled samples (each one a full solver run) before it can predict anything.

2. **PINN** learns from the governing PDEs plus a single reference operating point used to anchor the physical scales (linear power → flux magnitude and surface temperature). The learnable `phi_scale` together with the power-normalization loss is what couples the neutronics to the heat equation — without it the heat source is numerically zero and the two physics silently decouple.

3. The **tradeoff**: data-driven surrogates excel when a cheap solver can mass-produce training data; PINNs trade accuracy for data-independence and built-in physical consistency, and remain harder to optimize (loss balancing, spectral bias, interface discontinuities).

See the [README](../README.md) for the full mathematical formulation and references.
